#### Reproducing paper results

This code can reproduce the results presented in the paper. At present, it supports 40×40 genomaps in keeping with the original implementation.

**Important**: This means users have to select **exactly 1600 genes**.

First, let's import all the packages:

```python
# Import required packages

In [1]:
import torch
import warnings
import numpy as np
import sys
import os
sys.path.append(os.path.join(os.path.abspath(''), 'codes'))
# from examples.data_preprocessing.breast_cancer import preprocessing
import codes.gCCA.utils.preprocessing as genomap
from codes.gCCA.utils.train_gVAE import train_gVAE, GMM_fit
from codes.gCCA.utils.train_gCCA import train_gCCA
from codes.gCCA.utils.errorMetric import ComputeCorr
from codes.gCCA.utils.saveResults import plot_genomaps
from codes.gCCA.utils.train_ae import train_ae
import codes.gCCA.utils.batchMode as bm
import os

### Breast non batch mode

In [2]:
save_dir = './results/breast/non_batch'
# Create directory for saving results if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Check if GPU is available, or it may take a long time to run
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")

GPU available: True


#### Load data

- **scRNA_ref**: reference scRNA-seq anndata (cells x genes)
- **bulk_sample**: bulk RNA-seq anndata (samples x genes)  
- **patient_ids**: patient IDs
- **groundTruth**: ground truth proportions of cell types

Both `scRNA_ref` and `bulk_sample` contain the same genes, and are normalized to the sum of 1e4.

In [ ]:
#### Data preprocessing
# If you want to run the code on the raw data, please refer to the preprocessing function in the data_preprocessing folder:

# scRNA_ref, bulk_sample, groundTruth, annotations = preprocessing()

# You could skip the paragraph below and directly importing genes by running the code.
# If you want to select your own genes, here are the methods:



#### Gene Selection Methods
Genes can be selected using various approaches in scanpy. Here are the main options:

##### Method 1: Differential Expression Analysis
Use `rank_genes_groups` to identify differentially expressed genes:
```python
# Visualize top differentially expressed genes
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, key="wilcoxon")

# Extract top 400 genes from each group
deg = pd.DataFrame(adata.uns['wilcoxon']['names']).head(400)
deg_np = np.squeeze(deg.to_numpy().reshape(-1,1).astype(str))
```

##### Method 2: Highly Variable Genes
Use `highly_variable_genes` to select the most variable genes:

```python
# Select top 1600 highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=1600)

# Filter dataset to keep only highly variable genes
adata = adata[:, adata.var['highly_variable']]
```

> **Note**: To replicate the results of the paper, we fix the gene selection by directly loading a predefined gene set for reproducibility.

In [3]:
# Because the original data is more than 2 GB, we only provide the compressed data in the examples folder.

# Please unzip the scRNAref.h5ad in the folder '/data/examples/data/Wu_etal_2021/' before running the code. 

# If you run into an error saying 'scRNAref.h5ad' not exist, it's likely the file is not uncompressed.
warnings.filterwarnings('ignore')
seed = 21
import anndata as ad
# the scRNAref.h5ad exceed the upload size limit, so we provide the data in a compressed format.
# User can go to the below folder and unzip the file first.
scRNA_ref = ad.read_h5ad('./data/examples/data/Wu_etal_2021/scRNAref.h5ad')
bulk_sample = ad.read_h5ad('./data/examples/data/Wu_etal_2021/bulk.h5ad')
groundTruth = np.loadtxt('./data/examples/data/Wu_etal_2021/groundTruth.txt')
print('\033[93m Data loaded')

# Create genomaps from the scRNA-seq data
# Convert the scRNA-seq data to numpy array
genomap_ref = genomap.create(scRNA_ref, colNum=40, rowNum=40)
projMat = genomap_ref['projMat']
# save the genomap images
plot_genomaps(genomap_ref['genoMaps'], scRNA_ref.obs['cell_type'], save_dir=save_dir)
print('\033[93m Genomaps created')

 Data loaded
 Genomaps created


In [4]:
# Because scRNA is usually very sparse, so we create pseudobulks to train the gVAE (VAE for genomaps)
# gVAEs stores the trained VAE model, while pseudobulks stores the pseudobulk created from scRNA-seq data.
# When cell_subtype is not available, we use cell_type as the sub label, 
# this only ensures that the subtypes are well-mixed in pseudobulks
gVAEs, pseudobulks = train_gVAE(projMat,
                               scRNA_ref,
                               labels_main=scRNA_ref.obs['cell_type'],
                               labels_sub=scRNA_ref.obs['cell_subtype'], #can replace here with the same ['cell_type']
                               num_epochs = 1500
                               )
print('\033[93m gVAE training done')


 **This is in Training gVAE for cell type 0


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 5070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 11.37it/s, v_num=1, train_loss=2.260, recon=1.720, kld=0.541]

`Trainer.fit` stopped: `max_epochs=1500` reached.


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.84it/s, v_num=1, train_loss=2.260, recon=1.720, kld=0.541]
 **This is in Training gVAE for cell type 1


...

Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.55it/s, v_num=8, train_loss=2.160, recon=2.000, kld=0.156]
 **This is in Training gVAE for cell type 8


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 11.19it/s, v_num=9, train_loss=4.500, recon=2.120, kld=2.380]

`Trainer.fit` stopped: `max_epochs=1500` reached.


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.60it/s, v_num=9, train_loss=4.500, recon=2.120, kld=2.380]
 gVAE training done


In [5]:
# Train scikit-learn Gaussian Mixture model, save the mean and std for each cell type
torch.manual_seed(seed)
c_mean, c_std = GMM_fit(pseudobulks, projMat, gVAEs)

# Select a patient for decomposition
patient_id = 6
genoMap_bulk = genomap.convertGenomaps(bulk_sample.X, projMat,colNum=40, rowNum = 40)
# This is non batch mode, so the data are directly input into the gCCA model
# Here we only need the cell proportions, which is the second output of the train_gCCA function
_, paramsList_norm, _ = train_gCCA(genoMap_bulk[patient_id:patient_id+1], gVAEs, c_mean, c_std)

cell_proportions = np.array(paramsList_norm)

# Calculate correlations between the decomposed cell proportions and the ground truth
corr_gCCA = ComputeCorr(cell_proportions, groundTruth[patient_id:patient_id+1])
print('Correlations:', corr_gCCA)
print('Median correlation:', np.median(corr_gCCA))


with open(f'{save_dir}/results_nonbatch_mode.txt', 'w') as f:
    f.write(f'Patient ID: {patient_id}\n')
    f.write(f'Cell proportions correlation with ground truth:\n')
    f.write(f'{corr_gCCA}\n\n')
    f.write(f'Cell proportions:\n')
    f.write(f'{cell_proportions}\n')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 799: 100%|██████████| 1/1 [00:00<00:00, 60.37it/s, v_num=10, recon=8.410] 

`Trainer.fit` stopped: `max_epochs=800` reached.


Epoch 799: 100%|██████████| 1/1 [00:00<00:00, 48.63it/s, v_num=10, recon=8.410]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 60.41it/s, v_num=11, recon=4.150]

`Trainer.fit` stopped: `max_epochs=600` reached.


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 46.40it/s, v_num=11, recon=4.150]
Correlations: [0.7499515]
Median correlation: 0.7499514975174028


### Next is the Breast batch mode

In [ ]:
save_dir = './results/breast/batch'
# Create directory for saving results if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Load data
# scRNA_ref: reference scRNA-seq anndata (cells x genes)
# bulk_sample: bulk RNA-seq anndata (samples x genes)
# patient_ids: patient IDs
# groundTruth: ground truth proportions of cell types
# Both scRNA_ref and bulk_sample contain the same genes, and are normalized to the sum of 1e4.

# save the genomap images
plot_genomaps(genomap_ref['genoMaps'], scRNA_ref.obs['cell_type'], save_dir=save_dir)
print('\033[93m Genomaps created')
torch.manual_seed(720)
c_mean, c_std = GMM_fit(pseudobulks, projMat, gVAEs)

 Genomaps created


For batch mode, we assume all samples except the target sample are known,

so we first split the data into known and target samples.

#### For batch mode, we input the following: 
1. scRNA reference (genomaps) to generate pseudobulks 

2. scRNA labels (cell types)

3. training_bulk: the genomaps of the known samples

4. training_gt: the ground truth of the known samples

To avoid leaks of the testing set, this model needs to be trained separately for each testing sample

In [7]:

batch_correct_autoencoder = train_ae(scRNA_ref, genomap_ref, field_name='cell_type')
print('\033[93m autoencoder training done')

# Select a patient for decomposition
patient_id = 5

training_bulk, training_gt, testing_bulk, testing_gt = bm.Split(bulk_sample, projMat, groundTruth, patient_id)
batch_correction_model = bm.Train(genomap_ref, scRNA_ref, training_bulk, training_gt, batch_correct_autoencoder, field_name = 'cell_type')
print('\033[93m batch correction model training done')
corrected_bulk = bm.Correct(batch_correction_model, batch_correct_autoencoder, testing_bulk)

print('\033[93m This is in training for sample',patient_id)
_, cell_proportions , _ = train_gCCA(corrected_bulk, gVAEs, c_mean, c_std)


corr_gCCA = ComputeCorr(cell_proportions, groundTruth[patient_id:patient_id+1])
print('\033[93m Correlations:', corr_gCCA)
print('\033[93m Median correlation:', np.median(corr_gCCA))


with open(f'{save_dir}/results_batch_mode.txt', 'w') as f:
    f.write(f'Patient ID: {patient_id}\n')
    f.write(f'Cell proportions correlation with ground truth:\n')
    f.write(f'{corr_gCCA}\n\n')
    f.write(f'Cell proportions:\n')
    f.write(f'{cell_proportions}\n')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=200` reached.


 autoencoder training done


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=80` reached.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


 batch correction model training done
 This is in training for sample 5
Epoch 799: 100%|██████████| 1/1 [00:00<00:00, 57.19it/s, v_num=14, recon=2.570] 

`Trainer.fit` stopped: `max_epochs=800` reached.


...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 60.45it/s, v_num=15, recon=0.480]

`Trainer.fit` stopped: `max_epochs=600` reached.


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 46.40it/s, v_num=15, recon=0.480]
 Correlations: [0.97166901]
 Median correlation: 0.9716690140747428


### PBMC NON Batch

#### Similarly we use the PBMC dataset as another example

In [9]:
save_dir = './results/pbmc/non_batch'
# Create directory for saving results if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Check if GPU is available, or it may take a long time to run
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")

warnings.filterwarnings('ignore')
import anndata as ad
scRNA_ref = ad.read_h5ad('./data/examples/data/pbmc_real/scRNAref_nb.h5ad')
bulk_sample = ad.read_h5ad('./data/examples/data/pbmc_real/bulk_nb.h5ad')

print('\033[93m Data loaded')

# Create genomaps from the scRNA-seq data
# Convert the scRNA-seq data to numpy array
genomap_ref = genomap.create(scRNA_ref, colNum=40, rowNum=40)
projMat = genomap_ref['projMat']
plot_genomaps(genomap_ref['genoMaps'], scRNA_ref.obs['majortype_num'], save_dir=save_dir)
print('\033[93m genomaps created')


GPU available: True
 Data loaded
 genomaps created


Note here because cell_subtype is not available, we use cell_type as the sub label,

In [ ]:
gVAEs, pseudobulks = train_gVAE(projMat,
                               scRNA_ref,
                               labels_main=scRNA_ref.obs['majortype_num'],
                               labels_sub=scRNA_ref.obs['majortype_num'],
                               num_epochs=1500
                               )
 
# this only ensures that the subtypes are well-mixed in pseudobulks
print('\033[93m gVAE training done')

# Train scikit-learn Gaussian Mixture model
torch.manual_seed(720)
c_mean, c_std = GMM_fit(pseudobulks, projMat, gVAEs)

# Select a patient for decomposition to save time
patient_id = 0
# Convert bulk samples to genomaps and train gCCA
genoMap_bulk = genomap.convertGenomaps(bulk_sample.X, projMat, colNum=40, rowNum=40)
# Here we only need the cell proportions, which is the second output of the train_gCCA function
_, cell_proportions, _ = train_gCCA(genoMap_bulk[patient_id:patient_id+1], gVAEs, c_mean, c_std)
print('\033[93m Outputing results for patient', patient_id)
# The 5th column of the cell proportions is the 'OTHER' cell type (as preprocessed), which is not known in the ground truth
# So we set it to 0 (in the ground truth we already pre-set it to 0)
cell_proportions = np.delete(cell_proportions, 4, axis=1)
groundTruth = np.loadtxt('./data/examples/data/pbmc_real/groundTruth.txt')
groundTruth = np.delete(groundTruth, 4, axis=1)
corr_gCCA = ComputeCorr(cell_proportions, groundTruth[patient_id:patient_id+1])
print('The median correlation is:', np.median(corr_gCCA))



with open(f'{save_dir}/results_pbmc_nonbatch.txt', 'w') as f:
    f.write(f'Patient ID: {patient_id}\n')
    f.write(f'Cell proportions correlation with ground truth:\n')
    f.write(f'{corr_gCCA}\n\n')
    f.write(f'Cell proportions:\n')
    f.write(f'{cell_proportions}\n')

 **This is in Training gVAE for cell type 0


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 11.07it/s, v_num=16, train_loss=0.0521, recon=0.0521, kld=1.28e-6]

`Trainer.fit` stopped: `max_epochs=1500` reached.


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.49it/s, v_num=16, train_loss=0.0521, recon=0.0521, kld=1.28e-6]
 **This is in Training gVAE for cell type 1


...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 71.23it/s, v_num=24, recon=0.035] 

`Trainer.fit` stopped: `max_epochs=600` reached.


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 51.09it/s, v_num=24, recon=0.035]
The median correlation is: 0.9297038701373788


### Next is PBMC Batch mode

In [11]:
save_dir = './results/pbmc/batch'
# Create directory for saving results if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Check if GPU is available, or it may take a long time to run
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")

warnings.filterwarnings('ignore')

import anndata as ad
scRNA_ref = ad.read_h5ad('./data/examples/data/pbmc_real/scRNAref_b.h5ad')
bulk_sample = ad.read_h5ad('./data/examples/data/pbmc_real/bulk_b.h5ad')
groundTruth = np.loadtxt('./data/examples/data/pbmc_real/groundTruth.txt')
print('\033[93m Data loaded')


# Create genomaps from the scRNA-seq data
# Convert the scRNA-seq data to numpy array
genomap_ref = genomap.create(scRNA_ref, colNum=40, rowNum=40)
projMat = genomap_ref['projMat']
plot_genomaps(genomap_ref['genoMaps'], scRNA_ref.obs['majortype_num'], save_dir=save_dir)
print('\033[93m genomaps created')

GPU available: True
 Data loaded
 genomaps created


In [12]:
# Because scRNA is usually very sparse, so we create pseudobulks to train the gVAE (VAE for genomaps)
# gVAEs stores the trained VAE model, while pseudobulks stores the pseudobulk created from scRNA-seq data.
gVAEs, pseudobulks = train_gVAE(projMat,
                               scRNA_ref,
                               labels_main=scRNA_ref.obs['majortype_num'],
                               labels_sub=scRNA_ref.obs['majortype_num'],
                               num_epochs=1500
                               )
# when cell_subtype is not available, we use cell_type as the sub label, 
# this only ensures that the subtypes are well-mixed in pseudobulks
print('\033[93m gVAE training done')

# Train scikit-learn Gaussian Mixture model
torch.manual_seed(720)
c_mean, c_std = GMM_fit(pseudobulks, projMat, gVAEs)

# Select a patient for decomposition to save time
patient_id = 2
batch_correct_autoencoder = train_ae(scRNA_ref, genomap_ref,field_name = 'majortype_num')
training_bulk, training_gt, testing_bulk, testing_gt = bm.Split(bulk_sample, projMat, groundTruth, patient_id)
batch_correction_model = bm.Train(genomap_ref, scRNA_ref, training_bulk, training_gt, batch_correct_autoencoder)
print('\033[93m batch correction model training done')
corrected_bulk = bm.Correct(batch_correction_model, batch_correct_autoencoder, testing_bulk)
print('\033[93m This is in training for sample',patient_id)
_, cell_proportions, latentVars = train_gCCA(corrected_bulk, gVAEs, c_mean, c_std)

print('\033[93m Outputing results for patient', patient_id)
# The 5th column of the cell proportions is the 'OTHER' cell type (as preprocessed), which is not known in the ground truth
# So we set it to 0 (in the ground truth we already pre-set it to 0)
cell_proportions = np.delete(cell_proportions, 4, axis=1)
groundTruth = np.loadtxt('./data/examples/data/pbmc_real/groundTruth.txt')
groundTruth = np.delete(groundTruth, 4, axis=1)
corr_gCCA = ComputeCorr(cell_proportions, groundTruth[patient_id:patient_id+1])
print('The median correlation is:', np.median(corr_gCCA))



with open(f'{save_dir}/results_pbmc_batch.txt', 'w') as f:
    f.write(f'Patient ID: {patient_id}\n')
    f.write(f'Cell proportions correlation with ground truth:\n')
    f.write(f'{corr_gCCA}\n\n')
    f.write(f'Cell proportions:\n')
    f.write(f'{cell_proportions}\n')

 **This is in Training gVAE for cell type 0


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.98it/s, v_num=25, train_loss=0.733, recon=0.733, kld=0.000342]

`Trainer.fit` stopped: `max_epochs=1500` reached.


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.47it/s, v_num=25, train_loss=0.733, recon=0.733, kld=0.000342]
 **This is in Training gVAE for cell type 1


...

Epoch 799: 100%|██████████| 1/1 [00:00<00:00, 51.87it/s, v_num=34, recon=0.967]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 71.19it/s, v_num=35, recon=0.0368]

`Trainer.fit` stopped: `max_epochs=600` reached.


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 53.93it/s, v_num=35, recon=0.0368]
 Outputing results for patient 2
The median correlation is: 0.8928880433978678


### Finally, we use similar codes for ROSMAP dataset as the last example

#### Non batch mode

In [13]:
save_dir = './results/rosmap/non_batch'
# Create directory for saving results if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Check if GPU is available, or it may take a long time to run
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")

warnings.filterwarnings('ignore')

import anndata as ad
scRNA_ref = ad.read_h5ad('./data/examples/data/rosmap/scRNAref_nb.h5ad')
bulk_sample = ad.read_h5ad('./data/examples/data/rosmap/bulk_nb.h5ad')
groundTruth = np.loadtxt('./data/examples/data/rosmap/groundTruth.txt')

print('\033[93m Data loaded')



# Create genomaps from the scRNA-seq data
# Convert the scRNA-seq data to numpy array
genomap_ref = genomap.create(scRNA_ref, colNum=40, rowNum=40)
projMat = genomap_ref['projMat']
plot_genomaps(genomap_ref['genoMaps'], scRNA_ref.obs['celltype_num'], save_dir=save_dir)
print('\033[93m genomaps created')

# Because scRNA is usually very sparse, so we create pseudobulks to train the gVAE (VAE for genomaps)
# gVAEs stores the trained VAE model, while pseudobulks stores the pseudobulk created from scRNA-seq data.
gVAEs, pseudobulks = train_gVAE(projMat,
                               scRNA_ref,
                               labels_main=scRNA_ref.obs['celltype_num'],
                               labels_sub=scRNA_ref.obs['celltype_num'],
                               num_epochs=1500
                               )
# when cell_subtype is not available, we use cell_type as the sub label, 
# this only ensures that the subtypes are well-mixed in pseudobulks
print('\033[93m gVAE training done')

# Train scikit-learn Gaussian Mixture model
torch.manual_seed(42)
c_mean, c_std = GMM_fit(pseudobulks, projMat, gVAEs)
print('\033[93m Outputing results for patient', patient_id)
# Select a patient for decomposition to save time
patient_id = 1
# Convert bulk samples to genomaps and train gCCA
genoMap_bulk = genomap.convertGenomaps(bulk_sample.X, projMat, colNum=40, rowNum=40)
# Here we only need the cell proportions, which is the second output of the train_gCCA function
_, cell_proportions, _ = train_gCCA(genoMap_bulk[patient_id:patient_id+1], gVAEs, c_mean, c_std)
corr_gCCA = ComputeCorr(cell_proportions, groundTruth[patient_id:patient_id+1])
print('The median correlation is:', np.median(corr_gCCA))



with open(f'{save_dir}/results_rosmap_nonbatch.txt', 'w') as f:
    f.write(f'Patient ID: {patient_id}\n')
    f.write(f'Cell proportions correlation with ground truth:\n')
    f.write(f'{corr_gCCA}\n\n')
    f.write(f'Cell proportions:\n')
    f.write(f'{cell_proportions}\n')

GPU available: True
 Data loaded
 genomaps created
 **This is in Training gVAE for cell type 0


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 11.46it/s, v_num=36, train_loss=3.650, recon=3.650, kld=0.00117]

`Trainer.fit` stopped: `max_epochs=1500` reached.


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.87it/s, v_num=36, train_loss=3.650, recon=3.650, kld=0.00117]
 **This is in Training gVAE for cell type 1


...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 83.18it/s, v_num=42, recon=1.760] 

`Trainer.fit` stopped: `max_epochs=600` reached.


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 58.74it/s, v_num=42, recon=1.760]
The median correlation is: 0.7959421669011232


#### Batch mode

In [14]:
# Rosmap batch

save_dir = './results/rosmap/batch'
# Create directory for saving results if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Check if GPU is available, or it may take a long time to run
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")

warnings.filterwarnings('ignore')

import anndata as ad
scRNA_ref = ad.read_h5ad('./data/examples/data/rosmap/scRNAref_b.h5ad')
bulk_sample = ad.read_h5ad('./data/examples/data/rosmap/bulk_b.h5ad')
groundTruth = np.loadtxt('./data/examples/data/rosmap/groundTruth.txt')

print('\033[93m Data loaded')


# Create genomaps from the scRNA-seq data
# Convert the scRNA-seq data to numpy array
genomap_ref = genomap.create(scRNA_ref, colNum=40, rowNum=40)
projMat = genomap_ref['projMat']
plot_genomaps(genomap_ref['genoMaps'], scRNA_ref.obs['celltype_num'], save_dir=save_dir)
print('\033[93m genomaps created')

# Because scRNA is usually very sparse, so we create pseudobulks to train the gVAE (VAE for genomaps)
# gVAEs stores the trained VAE model, while pseudobulks stores the pseudobulk created from scRNA-seq data.
gVAEs, pseudobulks = train_gVAE(projMat,
                               scRNA_ref,
                               labels_main=scRNA_ref.obs['celltype_num'],
                               labels_sub=scRNA_ref.obs['celltype_num'],
                               num_epochs=1500
                               )
# when cell_subtype is not available, we use cell_type as the sub label, 
# this only ensures that the subtypes are well-mixed in pseudobulks
print('\033[93m gVAE training done')

# Train scikit-learn Gaussian Mixture model
torch.manual_seed(42)
c_mean, c_std = GMM_fit(pseudobulks, projMat, gVAEs)

# Select a patient for decomposition to save time
patient_id = 3
batch_correct_autoencoder = train_ae(scRNA_ref, genomap_ref,field_name = 'celltype_num')

training_bulk, training_gt, testing_bulk, testing_gt = bm.Split(bulk_sample, projMat, groundTruth, patient_id)
batch_correction_model = bm.Train(genomap_ref, scRNA_ref, training_bulk, training_gt, batch_correct_autoencoder, field_name = 'celltype_num')
print('\033[93m batch correction model training done')
corrected_bulk = bm.Correct(batch_correction_model, batch_correct_autoencoder, testing_bulk)
# Here we only need the cell proportions, which is the second output of the train_gCCA function
print('\033[93m This is in training for sample',patient_id)
print('\033[93m Outputing results for patient', patient_id)
_, cell_proportions , _ = train_gCCA(corrected_bulk, gVAEs, c_mean, c_std)
corr_gCCA = ComputeCorr(cell_proportions, groundTruth[patient_id:patient_id+1])
print('The median correlation is:', np.median(corr_gCCA))


with open(f'{save_dir}/results_rosmap_batch.txt', 'w') as f:
    f.write(f'Patient ID: {patient_id}\n')
    f.write(f'Cell proportions correlation with ground truth:\n')
    f.write(f'{corr_gCCA}\n\n')
    f.write(f'Cell proportions:\n')
    f.write(f'{cell_proportions}\n')


GPU available: True
 Data loaded
 genomaps created
 **This is in Training gVAE for cell type 0


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 11.13it/s, v_num=43, train_loss=5.290, recon=2.710, kld=2.580]

`Trainer.fit` stopped: `max_epochs=1500` reached.


Epoch 1499: 100%|██████████| 2/2 [00:00<00:00, 10.54it/s, v_num=43, train_loss=5.290, recon=2.710, kld=2.580]
 **This is in Training gVAE for cell type 1


...

Epoch 799: 100%|██████████| 1/1 [00:00<00:00, 66.53it/s, v_num=50, recon=0.826]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 89.45it/s, v_num=51, recon=0.133] 

`Trainer.fit` stopped: `max_epochs=600` reached.


Epoch 599: 100%|██████████| 1/1 [00:00<00:00, 65.84it/s, v_num=51, recon=0.133]
The median correlation is: 0.980861115746229
